In [ ]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer.primitives import EstimatorV2 as Estimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
import pandas as pd

n_samples = 2000  # subset — full 155K rows is impractical for QNN

# Shuffle before sampling so we get a mix of fire/no-fire rows
df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["target"].values.astype(float)

# split and scale
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

y_scaler = MinMaxScaler(feature_range=(-1, 1))
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

print(f"Train: {x_train_a.shape}, Test: {x_test_a.shape}")
print(f"y_train range: [{y_train_s.min():.3f}, {y_train_s.max():.3f}]")

Train: (1800, 9), Test: (200, 9)
y_train range: [-1.000, 1.000]


In [4]:
n_features = x_train_a.shape[1]
n_qubits = n_features

x_params = ParameterVector("x", n_features)
reps = 3
n_theta = (n_qubits)
theta_params = ParameterVector("θ", length=n_qubits * 2 * reps)

qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.ry(x_params[i], i)

t = 0
for r in range(reps):
    for q in range(n_qubits):
        qc.ry(theta_params[t], q); t += 1
        qc.rz(theta_params[t], q); t += 1

    for q in range(n_qubits):
        for r in range(q+1, n_qubits):
            qc.cx(q, r)

qc.decompose().draw("text")

global phase: (-0.5)*θ[1] - 0.5*θ[3] - 0.5*θ[5] - 0.5*θ[7] - 0.5*θ[9] - 0.5*θ[11] - 0.5*θ[13] - 0.5*θ[15] - 0.5*θ[17] - 0.5*θ[19] - 0.5*θ[21] - 0.5*θ[23] - 0.5*θ[25] - 0.5*θ[27] - 0.5*θ[29] - 0.5*θ[31] - 0.5*θ[33] - 0.5*θ[35] - 0.5*θ[37] - 0.5*θ[39] - 0.5*θ[41] - 0.5*θ[43] - 0.5*θ[45] - 0.5*θ[47] - 0.5*θ[49] - 0.5*θ[51] - 0.5*θ[53]
     ┌─────────────┐┌─────────────┐ ┌─────────┐                               »
q_0: ┤ R(x[0],π/2) ├┤ R(θ[0],π/2) ├─┤ P(θ[1]) ├───■────■─────────■─────────■──»
     ├─────────────┤├─────────────┤ ├─────────┤ ┌─┴─┐  │         │         │  »
q_1: ┤ R(x[1],π/2) ├┤ R(θ[2],π/2) ├─┤ P(θ[3]) ├─┤ X ├──┼────■────┼────■────┼──»
     ├─────────────┤├─────────────┤ ├─────────┤ └───┘┌─┴─┐┌─┴─┐  │    │    │  »
q_2: ┤ R(x[2],π/2) ├┤ R(θ[4],π/2) ├─┤ P(θ[5]) ├──────┤ X ├┤ X ├──┼────┼────┼──»
     ├─────────────┤├─────────────┤ ├─────────┤      └───┘└───┘┌─┴─┐┌─┴─┐  │  »
q_3: ┤ R(x[3],π/2) ├┤ R(θ[6],π/2) ├─┤ P(θ[7]) ├────────────────┤ X ├┤ X ├──┼──»
     ├─────────────┤├─────────────┤ ├─────────┤                └───┘└───┘┌─┴─┐»
q_4: ┤ R(x[4],π/2) ├┤ R(θ[8],π/2) ├─┤ P(θ[9]) ├──────────────────────────┤ X ├»
     ├─────────────┤├─────────────┴┐├─────────┴┐                         └───┘»
q_5: ┤ R(x[5],π/2) ├┤ R(θ[10],π/2) ├┤ P(θ[11]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_6: ┤ R(x[6],π/2) ├┤ R(θ[12],π/2) ├┤ P(θ[13]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_7: ┤ R(x[7],π/2) ├┤ R(θ[14],π/2) ├┤ P(θ[15]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_8: ┤ R(x[8],π/2) ├┤ R(θ[16],π/2) ├┤ P(θ[17]) ├──────────────────────────────»
     └─────────────┘└──────────────┘└──────────┘                              »
«                                                                           »
«q_0: ────────────■──────────────■───────────────────■───────────────────■──»
«                 │              │                   │                   │  »
«q_1: ───────■────┼─────────■────┼──────────────■────┼──────────────■────┼──»
«            │    │         │    │              │    │              │    │  »
«q_2: ──■────┼────┼────■────┼────┼─────────■────┼────┼─────────■────┼────┼──»
«     ┌─┴─┐  │    │    │    │    │         │    │    │         │    │    │  »
«q_3: ┤ X ├──┼────┼────┼────┼────┼────■────┼────┼────┼────■────┼────┼────┼──»
«     └───┘┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │    │    │    │    │  »
«q_4: ─────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼────┼────┼────┼────┼──»
«          └───┘┌─┴─┐└───┘┌─┴─┐  │  └───┘┌─┴─┐  │    │  ┌─┴─┐  │    │    │  »
«q_5: ──────────┤ X ├─────┤ X ├──┼───────┤ X ├──┼────┼──┤ X ├──┼────┼────┼──»
«               └───┘     └───┘┌─┴─┐     └───┘┌─┴─┐  │  └───┘┌─┴─┐  │    │  »
«q_6: ─────────────────────────┤ X ├──────────┤ X ├──┼───────┤ X ├──┼────┼──»
«                              └───┘          └───┘┌─┴─┐     └───┘┌─┴─┐  │  »
«q_7: ─────────────────────────────────────────────┤ X ├──────────┤ X ├──┼──»
«                                                  └───┘          └───┘┌─┴─┐»
«q_8: ─────────────────────────────────────────────────────────────────┤ X ├»
«                                                                      └───┘»
«     ┌──────────────┐┌──────────┐                                           »
«q_0: ┤ R(θ[18],π/2) ├┤ P(θ[19]) ├────────────────────────────────────────■──»
«     └──────────────┘└──────────┘          ┌──────────────┐┌──────────┐┌─┴─┐»
«q_1: ───────────────────────────────────■──┤ R(θ[20],π/2) ├┤ P(θ[21]) ├┤ X ├»
«                                        │  └──────────────┘└──────────┘└───┘»
«q_2: ──────────────────────────────■────┼────────────────────────────────■──»
«                                   │    │                                │  »
«q_3: ─────────────────────■────────┼────┼───────────────────────■────────┼──»
«                          │    

In [5]:
observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])

estimator = Estimator()

qnn = EstimatorQNN(
    circuit=qc,
    estimator=estimator,
    observables=observable,
    input_params=list(x_params),
    weight_params=list(theta_params),
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [6]:
torch.manual_seed(0)

model = TorchConnector(qnn)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# Use a small batch for quick testing; increase for full training
"""
n_train = 10000
xtr = torch.tensor(x_train_a[:n_train], dtype=torch.float32)
ytr = torch.tensor(y_train_s[:n_train], dtype=torch.float32).view(-1, 1)
"""
xtr = torch.tensor(x_train_a, dtype=torch.float32)
ytr = torch.tensor(y_train_s, dtype=torch.float32).view(-1,1)

for epoch in range(5):
    optimizer.zero_grad()
    yhat = model(xtr)
    loss = loss_fn(yhat, ytr)
    loss.backward()
    optimizer.step()
    print(f"epoch={epoch+1:3d}, loss={loss.item():.6f}")

epoch=  1, loss=1.218158
epoch=  2, loss=1.162502
epoch=  3, loss=1.103846
epoch=  4, loss=1.042033
epoch=  5, loss=0.973010


In [7]:
Xte = torch.tensor(x_test_a, dtype=torch.float32)
with torch.no_grad():
    ypred_s = model(Xte).numpy().ravel()

# Convert predictions back to original target units
ypred = y_scaler.inverse_transform(ypred_s.reshape(-1, 1)).ravel()

# Compare with original y_test (unscaled)
# Example metrics:
from sklearn.metrics import mean_squared_error, r2_score
print("MSE:", mean_squared_error(y_test, ypred))
print("R^2:", r2_score(y_test, ypred))

MSE: 0.41761851484422363
R^2: -173.71723069153714


In [14]:
from sklearn.metrics import roc_auc_score, roc_curve

# y_true: original regression target (not scaled), shape (n,)
# y_pred: your model prediction in the same units, shape (n,)

xte = torch.tensor(x_test_a, dtype=torch.float32)

with torch.no_grad():
    y_pred_scaled = model(xte).cpu().numpy().ravel()   # shape (n_test,)
# y_scaler is the MinMaxScaler you fit on y_train earlier
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
#y_pred = 1.8-y_pred

threshold = 0.65  # example: any burned area > 0 indicates a fire
y_true_bin = (y_test > threshold).astype(int)
auc = roc_auc_score(y_true_bin, y_pred)  # y_pred can be continuous
print("AUC_ROC:", auc)

AUC_ROC: 0.7035175879396984
